# Web scraping HKTV Mall Website On Beef Steaks

## Setup

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By # used for waiting page to load
from selenium.webdriver.support.ui import WebDriverWait # used for waiting page to load
from selenium.webdriver.support import expected_conditions as EC # used for waiting page to load
from bs4 import BeautifulSoup as bs
import time

url = "https://www.hktvmall.com/hktv/en/search_a?keyword=beef%20steak"
chrome_options = Options()
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)

## Set up variables

In [36]:
button = driver.find_element(By.XPATH, '//button[@id="sortMenu_next"]')
granditemlistName = []
granditemlistPrice = []
granditemlistSold = []

## Start scraping process

In [8]:
driver = webdriver.Chrome(options=chrome_options)
driver.get(url)
granditemlistName = []
granditemlistPrice = []
granditemlistSold = []
page_number = bs(driver.page_source).find_all('span', style="font-weight: 600;")
time.sleep(5)
print(page_number)
number = int(page_number[0].get_text().split(" ")[2])
for i in range(0, number):
    WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.CLASS_NAME, "brand-product-name")))
    items = bs(driver.page_source).find_all('div', class_= "brand-product-name")
    itemlistName = [i.text.strip() for i in items]
    items1 = bs(driver.page_source).find_all('div', class_= "price")
    itemlistPrice = [i.text.strip() for i in items1]
    items2 = bs(driver.page_source).find_all('div', class_= "salesNumber-container")
    itemlistSold = [i.text.strip() for i in items2]
    granditemlistName = granditemlistName + itemlistName
    granditemlistPrice = granditemlistPrice + itemlistPrice
    granditemlistSold = granditemlistSold + itemlistSold
    if i < number - 1:
        button = driver.find_element(By.XPATH, '//button[@id="sortMenu_next"]')
        button.click()

[<span style="font-weight: 600;">/  9 Pages</span>]


## Store variables

In [9]:
newitemlistPrice = granditemlistPrice
newitemlistName = granditemlistName
newitemlistSold = granditemlistSold

## Edit Variables

In [10]:
newitemlistPrice = granditemlistPrice
newitemlistPrice = [i.split(" ")[1] for i in newitemlistPrice]
newitemlistSold = [i.split("+")[0] for i in newitemlistSold]
neweritemlistSold = []
for i in newitemlistSold:
    if i == "":
        neweritemlistSold.append("0")
    else:
        neweritemlistSold.append(i)


## Create table

In [63]:
import pandas as pd
df = pd.DataFrame({
    'Product Name' : granditemlistName,
    'Price' : newitemlistPrice,
    'Number of sales' : neweritemlistSold
})
df["Number of sales"] = df["Number of sales"].apply(lambda x: str(x).replace(",","").replace(" ",""))
df["Price"] = pd.to_numeric(df["Price"], errors = "coerce")
df["Number of sales"] = pd.to_numeric(df["Number of sales"], errors = "coerce")

## Add Classes 

In [64]:
df["Steak Category"] = "Other"
df["Product Type"] = "Steak"
df["Wagyu?"] = "False"
df.loc[df['Product Name'].str.contains("Australia| AUS", case=False, na=False), 'Steak Category'] = 'Australian'
df.loc[df['Product Name'].str.contains(" United States| America", case=False, na= False), 'Steak Category'] = "American"
df.loc[df['Product Name'].str.contains("USA", case=True, na= False), "Steak Category"] = "American"
df.loc[df['Product Name'].str.contains("NZ| New Zealand", case= False, na= False), "Steak Category"] = "New Zealand"
df.loc[df["Product Name"].str.contains("Japan| JP", case = False, na = False), "Steak Category"] = 'Japanese'
df.loc[df['Product Name'].str.contains("Bar", case = False, na = False), "Product Type"] = "Bar"
df.loc[df['Product Name'].str.contains("Dog", case = False, na = False), "Product Type"] = "Dog Food"
df.loc[df['Product Name'].str.contains("Jerky", case = False, na = False), "Product Type"] = "Jerky"
df.loc[df['Product Name'].str.contains("Stick", case = False, na = False), "Product Type"] = "Stick"
df.loc[df["Product Name"].str.contains("Sauce", case = False, na = False), "Product Type"] = "Sauce"
df.loc[df['Product Name'].str.contains("Wagyu", case = False, na = False), "Wagyu?"] = "True"

### Section to display Table

In [65]:
df

,Product Name,Price,Number of sales,Steak Category,Product Type,Wagyu?
0,JACK LINK’S - Beef Steak Bar Char Grilled Orig...,52.0,100,Other,Bar,False
1,JACK LINK’S - Beef Steak Bar Pepper Flavour (3...,52.0,70,Other,Bar,False
2,JACK LINK’S - Beef Steak Bar Tasting Set 75g (...,54.0,40,Other,Bar,False
3,JACK LINK’S - Beef Steak Bar Teriyaki Flavour ...,52.0,50,Other,Bar,False
4,CJ - BEKSUL KR MARINADE SAUCE (KALBI) For Beef...,22.5,5000,Other,Sauce,False
...,...,...,...,...,...,...
532,Kikkoman - Japanese all-purpose teriyaki sauce...,10.0,20,Japanese,Sauce,False
533,JKOrganic - TOMC(Arcadian)Organic Striploin(25...,128.0,0,Other,Steak,False
534,JKOrganic - TOMC(Arcadian)Organic Striploin(25...,128.0,0,Other,Steak,False
535,"Mr.Meat - Christmas Combo Flash Deal ""Recipe V...",299.0,0,Other,Steak,False


### Section to save table 

In [16]:
datatableName = input("Table name")
df.to_excel(f"{datatableName}.xlsx")

Table name hi
hi hi


### Conclusion: which location produces the best steaks

In [67]:
tableofsteaks = df[df["Product Type"] == "Steak"]
groups = tableofsteaks.groupby("Steak Category")[["Number of sales", "Price"]].mean()

In [68]:
groups

,Number of sales,Price
Steak Category,,
American,372.222222,133.977778
Australian,197.368421,180.996316
Japanese,217.547170,219.096226
New Zealand,3967.777778,156.322222
Other,1665.647383,168.433799


### Conclusion: which location produces the most Wagyu steaks 

In [69]:
tableofsteaks = df[df["Product Type"] == "Steak"]
tableofsteaks.groupby("Steak Category")["Wagyu?"].count()

Series([], Name: Wagyu?, dtype: int64)

In [62]:
df[df["Steak Category"] == "New Zealand"]

,Product Name,Price,Number of sales,Steak Category,Product Type,Wagyu?
9,Yummy Land - New Zealand Grass-fed Beef Stripl...,102.0,NaN,New Zealand,Steak,False
283,全城熱賣 - [牛] ·⚡e⚡ New Zealand Hight Quality Stea...,129.9,90.0,New Zealand,Steak,False
302,ERIK’S MEATS - FROZEN NEW ZEALAND GRASS-FED SI...,65.0,NaN,New Zealand,Steak,False
303,SILVER FERN FARMS - New Zealand Boneless Beef ...,338.0,NaN,New Zealand,Steak,False
304,ERIK’S MEATS - FROZEN NEW ZEALAND GRASS-FED RI...,76.9,NaN,New Zealand,Steak,False
306,SILVER FERN FARMS - New Zealand Boneless Beef ...,302.0,900.0,New Zealand,Steak,False
309,SILVER FERN FARMS - New Zealand Boneless Beef ...,369.5,600.0,New Zealand,Steak,False
317,SILVER FERN FARMS - New Zealand Silver Fern Fa...,65.0,600.0,New Zealand,Steak,False
330,SILVER FERN FARMS - New Zealand Boneless Beef ...,450.5,100.0,New Zealand,Steak,False
344,SILVER FERN FARMS - New Zealand Grass-Fed Ribe...,98.0,300.0,New Zealand,Steak,False
